# Chapter 3 — Analysis and Transmission of Signals

Computer exercises from Section 3.10: computing Fourier transforms with the DFT/FFT, the time-shifting property, lowpass filtering, and autocorrelation / power spectral density of a random binary waveform.

```{admonition} Running these exercises
:class: tip
Every figure on this page is produced by the code directly above it, and the
code runs when the site is built. Use the **launch button** (the rocket icon) at the
top of the page to open this notebook in Google Colab or a live Binder session,
or hit **live code** to run and edit the cells right here in the browser.
```


## 3.10.1 Computing Fourier transforms with the DFT

The FFT gives us a numerical Fourier transform, but it is a transform of
*samples* of a signal, not of the signal itself. These two examples show how
close the numerical answer gets to the analytic one, and where it drifts.

Two cases, chosen because they stress different things:

- $g(t) = e^{-2t}u(t)$ — starts at $t = 0$ and decays forever, so truncating
  it at $T_0$ throws away a tail.
- $g(t) = \Pi(t/\tau)$ — starts at $t = -\tau/2$, so it straddles the origin
  and has to be wrapped around the FFT's periodic time axis.

In each figure the stems are the FFT result and the solid black line is the
exact transform, so any gap between them is numerical error.


### Example C3.1 — transform of $e^{-2t}u(t)$

The exact transform is

$$G(f) = \frac{1}{q + j2\pi f} \qquad \text{with } q = 2$$

One detail drives the whole result: the sample at $t = 0$ sits exactly on the
step's jump, so its value is taken as $1/2$ rather than $1$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import correlate

q = 2               # decay rate of the signal exp(-q t) u(t)
Ts = 1/64           # sampling interval
T0 = 4              # length of record we transform (the tail beyond this is lost)
N0 = int(T0 / Ts)   # number of samples

t = np.arange(N0) * Ts
# Multiplying by Ts turns the FFT's sum into an approximation of the integral
# that defines the Fourier transform.
g = Ts * np.exp(-q * t)

# t = 0 lands exactly on the step's discontinuity, so the sample takes the
# average of the two sides: half the value it would otherwise have.
g[0] = Ts * 0.5

Gnum = np.fft.fft(g)
Gp, Gm = np.angle(Gnum), np.abs(Gnum)

# FFT bin k corresponds to frequency k/(N0*Ts) Hz.
k = np.arange(N0)
fvec = k / (N0 * Ts)

# The exact transform, for comparison: G(f) = 1/(q + j*2*pi*f).
Gthy = 1 / (q + 1j * 2 * np.pi * fvec)

# Only the first 32 bins are plotted -- above that we are past the useful
# range and into the mirrored negative frequencies.
fig, axs = plt.subplots(2, 1, figsize=(8, 6))
axs[0].stem(fvec[:32], Gm[:32], linefmt='b')
axs[0].plot(fvec[:32], np.abs(Gthy)[:32], 'k-', linewidth=2)
axs[0].set_title('Magnitude');  axs[0].grid(True)
axs[0].set_xlabel(r'$f$ (Hz)')
axs[0].set_ylabel(r'$|G(f)|$')
axs[1].stem(fvec[:32], Gp[:32], linefmt='b')
axs[1].plot(fvec[:32], np.angle(Gthy)[:32], 'k-', linewidth=2)
axs[1].set_title('Phase');      axs[1].grid(True)
axs[1].set_xlabel(r'$f$ (Hz)')
axs[1].set_ylabel(r'$\theta_g(f)$ (rad)')
axs[1].set_ylim([-2,0])
fig.tight_layout(pad=1.0)
plt.show()


**Figure 1** — DFT of $g(t) = e^{-2t}u(t)$: magnitude and phase.
Stems are the FFT, the black curve is the exact $1/(q + j2\pi f)$. They track
closely at low frequency and separate as $f$ grows, where truncating the
signal's tail starts to matter.

### Example C3.2 — transform of a rectangular pulse

Now $g(t) = 8\,\Pi(t/\tau)$, whose exact transform is a sinc:

$$G(f) = A\tau\,\mathrm{sinc}(\pi f \tau)$$

This one is harder to sample correctly. The pulse is centred on $t = 0$, but
the FFT's time axis runs from $0$ to $T_0$, so the negative-time half of the
pulse has to be wrapped around to the end of the array. The code below builds
the first half, then mirrors it into the top of the array.

We reuse the `rect` helper from Chapter 2:


In [ ]:
def rect(t):
    """Unit rectangle: 1 on |t| < 0.5, 0 outside. Same helper as Chapter 2."""
    return np.array((np.sign(t+0.5) - np.sign(t-0.5)) > 0, dtype=float)


In [ ]:
tau = 1             # pulse width
B = 4/tau           # bandwidth we care about, roughly 4/tau Hz
Ts = 1/(2*B)        # sample at the Nyquist rate for that bandwidth
T0 = 4*tau          # length of record
N0 = int(T0/Ts)     # number of samples

k = np.arange(N0)
gtd = np.zeros(N0)
A = 8               # pulse amplitude

# The exact transform is a sinc. Note numpy's sinc is the NORMALISED one,
# sinc(x) = sin(pi x)/(pi x), which already includes the pi.
fsamp = np.linspace(-B, B, 4*N0+1)
Gf = A*tau*np.sinc(tau*fsamp)

# The pulse is centred on t = 0, but the FFT's time axis starts at t = 0.
# So we build the right-hand half here...
Tmid = int(np.ceil(N0/2))
gtd[:Tmid] = A*rect(np.arange(1,Tmid+1)*Ts/tau)

# ...and if a sample lands exactly on the pulse edge, give it the midpoint
# value A/2 rather than A or 0.
tedge = int(np.round(tau/(2*Ts)))
if abs(tau-tedge*2*Ts) < 1.e-13:
    gtd[tedge] = A/2

# ...then mirror that half into the TOP of the array, which is where the FFT
# expects negative time to live.
gtd[N0-1:N0-Tmid:-1] = gtd[1:Tmid]
tvec = k*Ts                         # scale time sampling instants in seconds
# fftshift moves the zero-frequency bin to the middle of the array, so the
# spectrum plots as -B ... 0 ... +B instead of 0 ... 2B.
Gq = np.real(np.fft.fftshift(np.fft.fft(Ts*gtd)))
fvec = k/(N0*Ts)-B

fig, axs = plt.subplots(2, 1, figsize=(8, 8))       # plot 2 periods
figtd1 = axs[0].stem(np.concatenate((tvec-T0, tvec)),
                np.concatenate((gtd, gtd)),
                    linefmt='b-', markerfmt='bo', basefmt=' ')
axs[0].axis([-4, 4, -0.2*A, A*1.2])
axs[0].set_xlabel(r'$t$ (sec.)');   axs[0].set_ylabel(r'$g_k$')
axs[0].set_title('Time response'); axs[0].grid()
figfd1 = axs[1].stem(fvec, Gq, linefmt='b-', markerfmt='bo', basefmt=' ')
figfd2 = axs[1].plot(fvec, Gq, 'b:')
axs[1].set_xlabel(r'$f$ (Hz)');     axs[1].set_ylabel(r'$G(f)$')
axs[1].set_title('Frequency response');   axs[1].grid()
# Plot analytical Fourier Transform on top of numerical values
figfd3 = axs[1].plot(fsamp, Gf, 'k')
plt.setp(figtd1, linewidth=2);  plt.setp(figfd1, linewidth=1)
plt.setp(figfd2, linewidth=2);  plt.setp(figfd3, linewidth=2)

fig.tight_layout(pad=1.0)
plt.show()


**Figure 2** — a rectangular pulse and its DFT. The stems follow the
sinc closely; the small ripple is the price of sampling a signal with a jump
discontinuity.

## 3.10.2 The time-shifting property

Delaying a signal does not change the magnitude of its spectrum — only the
phase, which picks up a linear term:

$$g(t - t_0) \;\longleftrightarrow\; G(f)\,e^{-j2\pi f t_0}$$

### Example C3.3

We first transform a triangular pulse, then delay it and watch only the phase
change. The triangle helper is the one from Chapter 2:


In [ ]:
def triangl(t):
    """Unit triangle: 1 - |t| on |t| < 1, 0 outside. Same helper as Chapter 2."""
    return np.array((1 - np.abs(t)) * ((t >= -1) & (t < 1)), dtype=float)


In [ ]:
# Transform of a triangular pulse, compared against the exact
# (A*tau/2) * sinc^2(pi f tau/2).
plt.clf(); plt.cla(); plt.close('all')

tau = 1
B = 4 / tau
Ts = 1 / (2 * B)
T0 = 4 * tau
N0 = int(T0/Ts)
k = np.arange(N0)
gtd = np.zeros(N0)
A = 4                   # pulse amplitude
fsamp = np.arange(4*N0+1)/(4*N0*Ts)-B # Select analytical samples in frequency
# Calculate Fourier transform analytically
Gf = A * tau * np.sinc(tau * fsamp / 2)**2/2  # Python sinc(x)=sin(pi x)/(pi x)
Tmid = int(np.ceil(N0 / 2))                   # Midpoint in FFT
gtd[:Tmid+1] = A*triangl(2*np.arange(0,Tmid+1)*Ts/tau)  # g(t) from 0 to Tmid
# Same wrap-around as the rectangular case: mirror the first half into the
# top of the array so the FFT sees a pulse centred on t = 0.
gtd[N0-1:N0-Tmid-1:-1] = gtd[1:Tmid+1]
tvec = k * Ts                                 # Time sampling instants
Gq = np.fft.fftshift(np.fft.fft(Ts * gtd))    # FFT and shift to center
Gm, Gp = np.abs(Gq), np.angle(Gq)             # Find magnitude/phase
fvec = k/(N0 * Ts) - B                        # Set the frequency range of FFT

plt.figure()        # Start to plot results
plt.subplot(211)
figtd1 = plt.stem(np.concatenate([tvec - T0, tvec]),
            np.concatenate([gtd, gtd]),
            linefmt='b-', markerfmt='bo') # 2 periods
plt.axis([-4, 4, -0.2 * A, A + 0.2 * A])
plt.title('Time response');     plt.grid()
plt.xlabel(r'${\it{t}}$ sec.'); plt.ylabel(r'${\it{g}}({\it{t}})$')
plt.tick_params(labelsize=10)

ax = plt.subplot(212)
figfd1 = plt.stem(fvec, Gm, linefmt='b-', markerfmt='bo') # Stem magnitude
figfd2 = plt.plot(fvec, Gm, 'b:') # replot magnitude
figfd3 = plt.plot(fsamp, Gf, 'k') # Analytic Fourier Transform
plt.title('Frequency response');  plt.grid()
plt.xlabel(r'${\it{f}}$ Hz');   plt.ylabel(r'${\it{G}}( {\it{f}})$')
plt.tick_params(labelsize=10)

plt.setp(figtd1, linewidth=2);  plt.setp(figfd1, linewidth=1)
plt.setp(figfd2, linewidth=2);  plt.setp(figfd3, linewidth=2)

plt.tight_layout(pad=1.0);    plt.show()


**Figure 3** — a triangular pulse and its DFT, against the exact
$\frac{A\tau}{2}\mathrm{sinc}^2(\pi f\tau/2)$. The triangle is smoother than
the rectangle, so its spectrum decays faster and the numerical fit is better.


In [ ]:
# Delay the SAME triangle by two samples and re-transform it. np.roll shifts
# the array circularly, which is exactly the delay the DFT assumes.
plt.figure(2, figsize=(8, 10))
gtdlay = np.roll(gtd, 2)

Gq = np.fft.fftshift(np.fft.fft(Ts * gtdlay)) # FFT and shift to center
Gm, Gp = np.abs(Gq), np.angle(Gq)             # Magnitude/phase
fvec=k / (N0 * Ts) - B          # Scaling the frequency range of FFT

# Theory says a delay of t0 multiplies the spectrum by exp(-j*2*pi*f*t0),
# which leaves |G(f)| alone and adds a phase ramp. Here t0 = 2*Ts.
Gfdlay = np.exp(-1j * 2 * np.pi * fvec * 2 * Ts)

plt.subplot(311)
figtd2_1 = plt.stem(np.concatenate([tvec-T0,tvec]), np.concatenate([gtdlay, gtdlay]),'b')
plt.axis([-4, 4, -0.2 * A, 1.2 * A])
plt.title('Time response');     plt.grid()
plt.xlabel('${\it{t}}$ sec.');  plt.ylabel('${\it{g}}({\it{t}})$')

ax = plt.subplot(312)
figfd2_1 = plt.stem(fvec, Gm, linefmt='b')
figfd2_2 = plt.plot(fvec, Gm, 'b:')
figfd2_3 = plt.plot(fsamp, Gf, 'k') # Analytical values
plt.title('Amplitude response');  plt.grid()
plt.xlabel('${\it{f}}$ Hz');    plt.ylabel('$|{\it{G}}( {\it{f}})|$')

ax = plt.subplot(313)
figfdp1 = plt.stem(fvec, Gp, linefmt='b')
figfdp2 = plt.plot(fvec, np.angle(Gfdlay))
plt.title('Phase response');    plt.grid()
plt.xlabel('${\it{f}}$ Hz');    plt.ylabel('$\Theta_{\it{g}}( {\it{f}})$')

plt.tick_params(labelsize=10)
plt.setp(figtd1, linewidth=2);  plt.setp(figfd1, linewidth=1)
plt.setp(figfd2, linewidth=2);  plt.setp(figfd3, linewidth=2)

plt.tight_layout(pad=1.0); plt.show()


**Figure 4** — the same triangle delayed by two samples,
$g(t - 2T_s)$. Compare the middle panel against Figure 3: the magnitude is
unchanged. All the delay information is in the bottom panel, where the phase
now falls linearly with frequency at a slope set by the delay.

## 3.10.3 Filtering

### Example C3.4

An ideal lowpass filter can be applied two ways, and this example does both so
you can see they agree:

1. **In frequency** — multiply $G(f)$ by the filter's gain $H(f)$, then inverse
   transform. Simple, but it needs the whole signal up front.
2. **In time** — convolve $g(t)$ with the filter's impulse response $h(t)$.
   This one can run sample-by-sample as the signal arrives.

The first cell rebuilds the rectangular pulse from Example C3.2 (the filtering
code needs its spectrum); the second does the filtering.


In [ ]:
# Rebuild the rectangular pulse and its spectrum from Example C3.2 -- the
# filtering code in the next cell needs Gq, fvec, gtd, N0 and Ts. This repeats
# the computation rather than the figure, so nothing is plotted here.
tau = 1
B = 4/tau
Ts = 1/(2*B)
T0 = 4*tau
N0 = int(T0/Ts)

k = np.arange(N0)
gtd = np.zeros(N0)
A = 8
fsamp = np.linspace(-B, B, 4*N0+1)
Gf = A*tau*np.sinc(tau*fsamp)

Tmid = int(np.ceil(N0/2))
gtd[:Tmid] = A*rect(np.arange(1,Tmid+1)*Ts/tau)
tedge = int(np.round(tau/(2*Ts)))
if abs(tau-tedge*2*Ts) < 1.e-13:
    gtd[tedge] = A/2
gtd[N0-1:N0-Tmid:-1] = gtd[1:Tmid]

tvec = k*Ts
Gq = np.real(np.fft.fftshift(np.fft.fft(Ts*gtd)))
fvec = k/(N0*Ts)-B


In [ ]:
# Filter g(t) = 8*rect(t/tau) two ways and show the answers agree:
#   (a) multiply by H(f) in the frequency domain, then inverse transform
#   (b) convolve with the impulse response h(t) in the time domain
# (a) needs the entire signal in hand; (b) could run as samples arrive.

q = np.arange(N0)
Tmid = np.ceil(N0/2).astype(int)

# Build an ideal brick-wall lowpass filter: gain 1 up to the cutoff, 0 above.
fcutoff = 2/tau                   # cutoff frequency, below the B = 4/tau limit
fs = 1/(N0 * Ts)                  # width of one FFT bin
qcutoff = np.ceil(fcutoff/fs).astype(int)
Hq = np.zeros(N0)
Hq[:qcutoff+1] = 1                # passband
Hq[qcutoff+1] = 0.5               # half gain exactly at cutoff
Hq[N0-1:N0-Tmid+1:-1] = Hq[1:Tmid-1]   # mirror for negative frequencies

# METHOD (a): filtering is multiplication in the frequency domain.
# Dividing by Ts undoes the scaling applied when Gq was built.
Yq = Gq * np.fft.fftshift(Hq)
yk = np.fft.ifft(np.fft.fftshift(Yq))/Ts

# METHOD (b): the same filter as a convolution in the time domain. h(t) is the
# inverse transform of H(f) -- a sinc, since H(f) is a rectangle.
gtshift = np.fft.fftshift(gtd)
hk = np.fft.fftshift(np.fft.ifft(Hq))
ykconv = np.convolve(gtshift, hk)

# Convolution output is longer than the input, so pad (a) to match for plotting.
ykpad = np.concatenate([np.zeros(N0//2), np.fft.fftshift(yk), np.zeros(N0//2-1)])

plt.figure(figsize=(8,12))
plt.subplot(311);  plt.grid()
figfd1 = plt.stem(fvec,Gq, linefmt='b') # Gq of rect(t/tau)
plt.title(r'Input frequency response of $A \cdot rect(t / \tau )$')
plt.xlabel('${\it{f}}$ Hz');    plt.ylabel('${\it{G}}( {\it{f}})$')

plt.subplot(312); plt.grid()
figfd2 = plt.stem(fvec, np.fft.fftshift(Hq), linefmt='b') # ideal LPF H(f)
plt.title('Lowpass filter gain')
plt.xlabel('${\it{f}}$ Hz');    plt.ylabel('${\it{H}}( {\it{f}})$')

ax = plt.subplot(313);  plt.grid()
figtd1 = plt.stem(np.arange(2 * N0 - 1)*Ts, np.real(ykpad),'k', label='FFT') # LPF output
figtd2 = plt.plot(np.arange(2 * N0 - 1)*Ts, np.real(ykconv), 'b', label='Convolution')
plt.title('Lowpass filter outputs')
plt.xlabel('${\it{t}}$ sec.');  plt.ylabel('${\it{y}}({\it{t}})$')
plt.axis([0, 8, -0.2 * A, 1.2 * A]);    plt.legend()

#plt.tick_params(labelsize=10)
plt.setp(figtd1, linewidth=1); plt.setp(figtd2, linewidth=1)
plt.setp(figfd1, linewidth=1); plt.setp(figfd2, linewidth=1)

plt.tight_layout(pad=1.0); plt.show()


**Figure 5** — lowpass filtering of a rectangular pulse: the input
spectrum, the filter gain, and the output computed both ways. The two output
traces lie on top of each other, which is the point. The overshoot and ringing
at the pulse edges is Gibbs' phenomenon — the price of cutting the spectrum off
sharply.

## 3.10.4 Autocorrelation and power spectral density

### Example C3.5

A random binary waveform has no Fourier transform — it never repeats and never
settles. What it does have is an autocorrelation function

$$R_g(\tau) = \overline{g(t)\,g(t+\tau)}$$

whose Fourier transform is the power spectral density. That is the route this
example takes: generate 20,000 random bits, measure $R_g(\tau)$, then FFT it.


In [ ]:
# A random binary waveform has no Fourier transform -- it never repeats. What
# it has is an autocorrelation function, and the transform of THAT is the
# power spectral density. Steps: build the waveform, measure R_g(tau), FFT it.
np.random.seed(447)       # fixed seed so the published figure is reproducible

tau = 0.1                 # pulse width
B = 4 / tau               # frequency range of interest
Ts = 1 / (2 * B)          # sampling interval
tduty = 0.5               # duty cycle: the pulse fills half of each bit slot
Tb = tau / tduty          # bit period
Np = int(Tb / Ts)         # samples per bit
Nfft = 256                # FFT length for the PSD
A = 1                     # pulse amplitude

# One return-to-zero pulse: on for the first half of the bit slot, off after.
puls = np.zeros(Np)
puls[:Np] = A * rect(np.arange(1,Np+1)*Ts/tau-0.5)

# 20,000 random bits, mapped to -1 and +1 with equal probability.
Na = 20000
a = 2 * np.random.randint(2, size=(Na,)) - 1

# Place one data value at the start of each bit slot and zeros elsewhere, then
# convolve with the pulse shape -- that stamps a copy of the pulse onto each
# slot, which is exactly what a baseband transmitter does.
s = np.stack([a] + [a*0] * (Np-1), axis=-1).reshape(-1)
gt = np.convolve(s, puls, mode='full')

# Autocorrelation, kept to +/- 2 bit periods of lag (beyond that it is zero:
# separate bits are independent). Dividing by len(gt) makes it an average.
lagcorrfunc = 2
lagsamp = lagcorrfunc * Np
rgtau = correlate(gt, gt, 'full', method='fft')[
    len(gt)-1-lagsamp:len(gt)+lagsamp] / len(gt)
tauvec = np.arange(-lagsamp, lagsamp+1) * Ts

# Zero-pad to Nfft for a smoother spectrum, then rotate so lag 0 sits at
# index 0 -- the FFT expects its origin there, not in the middle.
rgpad = np.concatenate((rgtau, np.zeros(Nfft-len(rgtau))))
rgshift = np.roll(rgpad, -lagsamp)

# Wiener-Khinchin: the PSD is the Fourier transform of the autocorrelation.
PowerSD = np.fft.fft(rgshift) * Ts
fvec = np.arange(Nfft) / (Ts * Nfft) - B

fig, axs = plt.subplots(nrows=2, ncols=1)
fig.suptitle('Autocorrelation and PSD')
figcor = axs[0].plot(tauvec, rgtau,'purple')
axs[0].set_title('Autocorrelation function');   axs[0].grid(True)
axs[0].set_xlabel(r'$ \tau $ sec.');
axs[0].set_ylabel(r'$ R_{\it{g}}( \tau ) $')

figpsd = axs[1].plot(fvec, np.fft.fftshift(np.real(PowerSD)),'purple')
axs[1].set_title('PSD');  axs[1].grid(True)
axs[1].set_xlabel('${\it{f}}$ Hz')
axs[1].set_ylabel('${\it{S}}_{\it{g}}({\it{f}})$')

plt.tick_params(labelsize=10)
plt.setp(figcor, linewidth=2);  plt.setp(figpsd, linewidth=2)
plt.tight_layout(pad=1.0); plt.show()


**Figure 6** — autocorrelation and PSD of a random binary
waveform. $R_g(\tau)$ peaks sharply at $\tau = 0$ and falls away within one
pulse width, because separate bits are independent. Its transform is the
familiar $\mathrm{sinc}^2$ power spectrum, whose first null sits at $1/\tau$ —
which is why the pulse width sets the bandwidth the signal needs.
